## 실험 1. 선형 모델의 Sharpe Ratio 비교
**Point** <br>
- 주 타깃 **`excess`** (국채 대비 초과수익, 연속). 보조로 `spread_positive`(이진), `bad`(부도).
- 평가 **동일가중 샤프**. 거부 건은 초과수익 0(전액 국채)으로 포함.

**목차**

1. Sharpe 계산 함수 정의: `sharpe()`, `sweep_threshold()` <br>
2. Logistic Regression 모델 정의 및 Baseline 성능 평가
3. k-means + Linear Regression 모델 정의 및 Baseline 성능 평가
4. 하이퍼파라미터 튜닝 및 최종 모델 확정
5. 선형 모델 최종 비교 <br>
6. 부트스트래핑: 확정 모델 간 Sharpe 차이의 통계적 유의성 검증 <br>
7. 성능 평가: 최종 모델 sanity check


### 0. 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 데이터 로드
df = pd.read_csv("model_input_linear.csv")

# 입력 금지 변수 모음
NOT_FEATURE = ["id", "split", "grade", "term_n", "funded_amnt", "rf", "irr",
               "excess", "excess_contract", "spread_positive", "bad"]

feature_cols = [c for c in df.columns if c not in NOT_FEATURE]

train = df[df["split"] == "train"]
val   = df[df["split"] == "val"]

X_train = train[feature_cols]
X_val   = val[feature_cols]

print(f"feature 수: {len(feature_cols)}")  

feature 수: 47


### 1. Sharpe 계산 함수 정의
- `sharpe()`: 예측값 상위 pct%만 승인, 거부 건은 excess=0(국채) 처리 후 동일가중 Sharpe 계산
- `sweep_threshold()`: pct를 스윕하며 Sharpe 최대 지점 탐색
이후 모든 모델의 공통 평가 함수로 사용

In [2]:
def sharpe(pred, excess, pct):
    """예측값 상위 pct% 승인, 거부 건은 excess=0(국채)"""
    thr = np.quantile(pred, 1 - pct)
    approve = pred >= thr
    x = np.where(approve, excess, 0.0)
    sd = x.std(ddof=1)
    return 0.0 if sd == 0 else x.mean() / sd

def sweep_threshold(pred_val, excess_val, pcts=np.arange(0.05, 0.95, 0.05)):
    """pct별 Sharpe를 표로"""
    rows = []
    for pct in pcts:
        s = sharpe(pred_val, excess_val, pct)
        rows.append({"pct": round(pct, 2), "sharpe": s})
    result = pd.DataFrame(rows)
    best = result.loc[result["sharpe"].idxmax()]
    print(f"최고 Sharpe: {best['sharpe']:.4f} (승인률 {best['pct']*100:.0f}%)")
    return result, best

### 2. Logistic Regression (y = spread_positive, 확률 예측)
이진 타깃 `spread_positive`(초과수익이 양수인지 여부)를 확률로 예측. <br>
평가 기준이 val Sharpe이므로, 예측 확률을 크기순 정렬해 상위 pct%만 승인했다고 가정하고 `sweep_threshold()`로 pct를 스윕하며 Sharpe가 최대화되는 지점 탐색.

In [3]:
y_train_bin = train["spread_positive"]

logit = LogisticRegression(max_iter=1000)
logit.fit(X_train, y_train_bin)

pred_logit = logit.predict_proba(X_val)[:, 1]   # 확률값

excess_val = val["excess"].values
result_logit, best_logit = sweep_threshold(pred_logit, excess_val)

최고 Sharpe: 0.0983 (승인률 85%)


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 3. k-means + Linear Regression (y = excess, 연속 예측)
- 연속 타깃 `excess`(국채 대비 초과수익)를 직접 회귀로 예측
- 표본을 K-means로 k개 군집으로 나눈 뒤, 군집별로 별도의 Ridge 회귀식을 적합: 단일 회귀식이 못 잡는 군집 간 이질적 패턴을 반영하기 위함
- 4절에서 Optuna로 k·alpha를 함께 튜닝해 최종 모델 확정

In [7]:
y_train_cont = train["excess"]

# 3-1. train으로 클러스터링 (스케일링 후)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
train_cluster = kmeans.fit_predict(X_train_scaled)
val_cluster   = kmeans.predict(X_val_scaled)

# 3-2. 클러스터별 Linear Regression
cluster_models = {}
for k in range(5):
    mask = train_cluster == k
    m = LinearRegression()
    m.fit(X_train[mask], y_train_cont[mask])
    cluster_models[k] = m
    print(f"cluster {k}: train n={mask.sum()}")

# 3-3. val 예측 (각 행이 속한 클러스터 모델로)
pred_5means = np.zeros(len(X_val))
for k in range(5):
    mask = val_cluster == k
    if mask.sum() > 0:
        pred_5means[mask] = cluster_models[k].predict(X_val[mask])

result_5means, best_5means = sweep_threshold(pred_5means, excess_val)

cluster 0: train n=18357
cluster 1: train n=66670
cluster 2: train n=116976
cluster 3: train n=139283
cluster 4: train n=89495
최고 Sharpe: 0.1684 (승인률 45%)


### 4. 하이퍼파라미터 튜닝
앞선 2·3절의 개별 실험은 방향성 확인용. 최종 확정 파라미터는 본 절의 튜닝 결과를 따른다. <br>

- LR: C(정규화 강도)·penalty(L1/L2)·class_weight 함께 탐색 (스케일링 포함) <br>
- K-means+Ridge: k(군집 수)·alpha(Ridge 정규화 강도) 함께 탐색 <br>
K-means+Ridge는 train/val 클러스터 비율 불일치가 크면 페널티를 줘서, 불안정한 클러스터링으로 쏠리지 않게 함

In [ ]:
import warnings
import numpy as np
import optuna
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

# 재현성
SEED = 42
N_TRIALS = 40 

# 1. Logistic Regression 튜닝
def objective_lr(trial):
    C = trial.suggest_float("C", 1e-4, 1e2, log=True)
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    solver = "liblinear" if penalty == "l1" else "lbfgs"

    pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C, penalty=penalty, solver=solver,
            class_weight=class_weight, max_iter=2000,
        )
    )
    pipe.fit(X_train, y_train_bin)
    pred = pipe.predict_proba(X_val)[:, 1]
    _, best = sweep_threshold(pred, excess_val, verbose=False)

    print(f"[Trial {trial.number:2d}] C={C:.2e} penalty={penalty} "
          f"cw={class_weight} → sharpe={best['sharpe']:.4f} (pct={best['pct']})")
    return best["sharpe"]


print("=" * 60)
print("[LR Optuna 튜닝]")
print("=" * 60)
study_lr = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study_lr.optimize(objective_lr, n_trials=N_TRIALS)
print("\n최적 파라미터:", study_lr.best_params)
print("최고 Sharpe:", study_lr.best_value)

# 확정 모델 재학습 + val 재검증
best_p = study_lr.best_params
solver = "liblinear" if best_p["penalty"] == "l1" else "lbfgs"
final_lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=best_p["C"], penalty=best_p["penalty"], solver=solver,
        class_weight=best_p["class_weight"], max_iter=2000,
    )
)
final_lr.fit(X_train, y_train_bin)
pred_lr_final = final_lr.predict_proba(X_val)[:, 1]
print("\n[LR 최종 확정 모델 검증]")
_, best_lr_final = sweep_threshold(pred_lr_final, excess_val)


# 2. 5-means + Ridge 재튜닝 (LinearRegression → Ridge로 일반화, alpha가 0에 가까우면 기존 LinearRegression과 사실상 동일)
def objective_kmeans(trial):
    k = trial.suggest_int("k", 3, 8)
    alpha = trial.suggest_float("alpha", 1e-3, 1e2, log=True)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    train_cluster = kmeans.fit_predict(X_train_scaled)
    val_cluster = kmeans.predict(X_val_scaled)

    pred = np.zeros(len(X_val))
    cluster_ratio_diff = []
    for c in range(k):
        mask_tr = train_cluster == c
        mask_va = val_cluster == c
        if mask_tr.sum() == 0 or mask_va.sum() == 0:
            continue
        m = Ridge(alpha=alpha)
        m.fit(X_train[mask_tr], y_train_cont[mask_tr])
        pred[mask_va] = m.predict(X_val[mask_va])

        # train/val 클러스터 비율 불일치 체크 
        ratio_tr = mask_tr.sum() / len(X_train)
        ratio_va = mask_va.sum() / len(X_val)
        cluster_ratio_diff.append(abs(ratio_tr - ratio_va))

    _, best = sweep_threshold(pred, excess_val, verbose=False)
    max_diff = max(cluster_ratio_diff) if cluster_ratio_diff else 1.0

    print(f"[Trial {trial.number:2d}] k={k} alpha={alpha:.2e} "
          f"→ sharpe={best['sharpe']:.4f} (pct={best['pct']}) "
          f"max_ratio_diff={max_diff*100:.2f}%p")

    # 비율 불일치가 심하면 페널티를 줘서 Optuna가 불안정한 클러스터링으로 쏠리지 않게 함
    trial.set_user_attr("max_ratio_diff", max_diff)
    return best["sharpe"]


print("\n" + "=" * 60)
print("[5-means+Ridge Optuna 튜닝]")
print("=" * 60)
study_km = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study_km.optimize(objective_kmeans, n_trials=N_TRIALS)
print("\n최적 파라미터:", study_km.best_params)
print("최고 Sharpe:", study_km.best_value)
print("해당 trial의 max_ratio_diff:",
      study_km.best_trial.user_attrs.get("max_ratio_diff"))

# best_trial의 max_ratio_diff가 크면(예: >5%p) train/val 불안정 가능성 
print("\n상위 5개 trial (sharpe 기준, ratio_diff 함께 확인):")
trials_sorted = sorted(study_km.trials, key=lambda t: t.value, reverse=True)[:5]
for t in trials_sorted:
    print(f"  trial {t.number}: sharpe={t.value:.4f}, "
          f"params={t.params}, ratio_diff={t.user_attrs.get('max_ratio_diff'):.4f}")

[LR Optuna 튜닝]
[Trial  0] C=1.77e-02 penalty=l1 cw=None → sharpe=0.1626 (pct=0.55)
[Trial  1] C=8.63e-04 penalty=l2 cw=balanced → sharpe=0.1623 (pct=0.55)
[Trial  2] C=1.33e-04 penalty=l1 cw=None → sharpe=0.1407 (pct=0.55)
[Trial  3] C=1.26e-03 penalty=l2 cw=None → sharpe=0.1622 (pct=0.55)
[Trial  4] C=4.69e-01 penalty=l2 cw=balanced → sharpe=0.1618 (pct=0.4)
[Trial  5] C=5.14e+00 penalty=l2 cw=None → sharpe=0.1621 (pct=0.55)
[Trial  6] C=4.42e-01 penalty=l1 cw=balanced → sharpe=0.1619 (pct=0.4)
[Trial  7] C=7.09e+00 penalty=l1 cw=None → sharpe=0.1620 (pct=0.55)
[Trial  8] C=5.40e-04 penalty=l1 cw=None → sharpe=0.1574 (pct=0.55)
[Trial  9] C=9.44e-01 penalty=l2 cw=None → sharpe=0.1620 (pct=0.55)
[Trial 10] C=2.98e-02 penalty=l1 cw=balanced → sharpe=0.1620 (pct=0.55)
[Trial 11] C=1.03e-02 penalty=l2 cw=balanced → sharpe=0.1620 (pct=0.4)
[Trial 12] C=1.73e-02 penalty=l1 cw=balanced → sharpe=0.1621 (pct=0.55)
[Trial 13] C=3.26e-03 penalty=l2 cw=balanced → sharpe=0.1619 (pct=0.5)
[Trial 14

#### 4-1. Logistic Regression의 튜닝 결과
- 최고 Sharpe: 0.1627 (승인률 55%)
- 최적 파라미터: {'C': 0.0021732389322010577, 'penalty': 'l2', 'class_weight': None} 
- 최고 Sharpe: 0.16270285357130562 

#### 4-2. K-Means + Linear Regression의 튜닝 결과
- 최고 Sharpe: 0.17166955548525953  
- 최적 파라미터: {'k': 7, 'alpha': 12.867193557921818} 
- 해당 trial의 max_ratio_diff: 0.001802571586247692


### 5. 선형 모델 간 비교

| 모델 | 세부 설정 | Sharpe | 승인률(threshold pct)|
|---|---|---|---|
| Logistic Regression | L1, C=0.01, no weight | 0.1627 | 55% |
| **K-Means + Linear** | **k=7, Ridge** | **0.1716** | **40%** |

### 6. 부트스트래핑
- LR과 K-means+Ridge, 두 확정 모델(4절 튜닝 결과)의 val Sharpe 차이가 우연인지 통계적으로 유의한지 검증
- 방식 A: 각자 최적 승인율(LR 55% vs K-means+Ridge 40%)에서 비교
- 방식 B: 승인율을 40%로 고정하고 비교 — 이 40%는 K-means+Ridge 자체의 최적 승인율이며, 최종 챔피언 모델(CatBoost)의 45% 컷오프와는 별개
- 두 방식 모두 500회(n_boot=1000, 재표본 1000회) 부트스트랩으로 95% 신뢰구간 계산, CI가 0을 포함하지 않으면 유의함

In [ ]:
def bootstrap_compare(pred_a, pct_a, pred_b, pct_b, excess, n_boot=1000, seed=42):
    """모델 B - 모델 A의 Sharpe 차이를 부트스트랩으로 검증.
    lo, hi 95% CI가 0을 포함하지 않으면 유의함."""
    rng = np.random.default_rng(seed)
    n = len(excess)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        s_a = sharpe(pred_a[idx], excess[idx], pct_a)
        s_b = sharpe(pred_b[idx], excess[idx], pct_b)
        diffs.append(s_b - s_a)
    diffs = np.array(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    point_diff = sharpe(pred_b, excess, pct_b) - sharpe(pred_a, excess, pct_a)
    print(f"차이 (B − A): {point_diff:+.4f}")
    print(f"95% CI: [{lo:+.4f}, {hi:+.4f}]")
    print(f"유의성: {'유의함 (0 미포함)' if lo * hi > 0 else '불확실 (0 포함)'}")
    return diffs

In [ ]:
"""
최종 확정 모델(재튜닝 결과)로 부트스트랩 유의성 검증 재실행
- LR: C=0.0021732389322010577, penalty='l2', class_weight=None
- k-means+Ridge: k=7, alpha=12.867193557921818
"""

import numpy as np
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


# 1. 확정 모델 A: Logistic Regression 
final_logit = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.0021732389322010577, penalty="l2", max_iter=2000)
)
final_logit.fit(X_train, y_train_bin)
pred_logit_final = final_logit.predict_proba(X_val)[:, 1]

print("[LR 확정 모델 검증]")
_, best_logit = sweep_threshold(pred_logit_final, excess_val)


# 2. 확정 모델 B: k-means(k=7) + Ridge 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

kmeans7 = KMeans(n_clusters=7, random_state=42, n_init=10)
train_cluster = kmeans7.fit_predict(X_train_scaled)
val_cluster   = kmeans7.predict(X_val_scaled)  

cluster_models = {}
for c in range(7):
    mask = train_cluster == c
    m = Ridge(alpha=12.867193557921818)
    m.fit(X_train[mask], y_train_cont[mask])
    cluster_models[c] = m

pred_5means_final = np.zeros(len(X_val))
for c in range(7):
    mask = val_cluster == c
    if mask.sum() > 0:
        pred_5means_final[mask] = cluster_models[c].predict(X_val[mask])

print("\n[k-means(k=7)+Ridge 확정 모델 검증]")
_, best_5means = sweep_threshold(pred_5means_final, excess_val)


# 3. 부트스트랩 비교 
print("\n=== 방식 A: 각자 최적 승인률 (LR 55% vs k-means 40%) ===")
diffs_a = bootstrap_compare(pred_logit_final, best_logit["pct"],
                             pred_5means_final, best_5means["pct"], excess_val)

# 방식 B의 고정 pct는 두 모델의 최적 pct 중 더 자주 나온 쪽으로.
# 이번엔 LR=55%, k-means=40%로 서로 다르므로, 두 값의 중간 지점인 45%나
# 팀에서 실험효과분해에 합의한 pct=0.40 중 하나로 통일 권장.
print("\n=== 방식 B: 승인률 고정 40% (실험 설계 pct=0.40과 통일) ===")
diffs_b = bootstrap_compare(pred_logit_final, 0.40,
                             pred_5means_final, 0.40, excess_val)

[LR 확정 모델 검증]
최고 Sharpe: 0.1627 (승인률 55%)

[5-means(k=7)+Ridge 확정 모델 검증]
최고 Sharpe: 0.1717 (승인률 40%)

=== 방식 A: 각자 최적 승인률 (LR 55% vs 5-means 40%) ===
차이 (B − A): +0.0090
95% CI: [+0.0026, +0.0154]
유의성: 유의함 (0 미포함)

=== 방식 B: 승인률 고정 40% (실험 설계 pct=0.40과 통일) ===
차이 (B − A): +0.0110
95% CI: [+0.0044, +0.0171]
유의성: 유의함 (0 미포함)


---
## 성능 평가
확정된 K-means+Ridge 모델(k=7)의 val Sharpe를 최종적으로 다시 한번 확인하는 sanity check 단계.

In [26]:
print(sweep_threshold(pred_5means_final, excess_val))

최고 Sharpe: 0.1717 (승인률 40%)
(     pct    sharpe
0   0.05  0.082587
1   0.10  0.106692
2   0.15  0.126260
3   0.20  0.142231
4   0.25  0.154256
5   0.30  0.162021
6   0.35  0.167772
7   0.40  0.171670
8   0.45  0.169758
9   0.50  0.169286
10  0.55  0.166550
11  0.60  0.163961
12  0.65  0.161319
13  0.70  0.158708
14  0.75  0.150746
15  0.80  0.141991
16  0.85  0.134697
17  0.90  0.122773, pct       0.40000
sharpe    0.17167
Name: 7, dtype: float64)


4절·6절에서 이미 확인한 Sharpe(0.1717, 승인율 40%)와 동일하게 재현됨을 확인하였다. 앞선 모든 단계(스케일링, 클러스터링, Ridge 재학습)가 일관되게 적용되었음을 보여주는 최종 검증.